# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List the record sets and their fields with `@id`s and names

print("Available record sets:")
for rs in metadata.record_sets:
    print(f"  RecordSet @id: {rs.id}, name: {getattr(rs, 'name', None)}")
    print("    Fields:")
    for field in rs.fields:
        print(f"      Field @id: {field.id}, name: {getattr(field, 'name', None)}, dataType: {getattr(field, 'data_type', None)}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from each record set
record_sets = [rs.id for rs in metadata.record_sets]
dataframes = {}

for record_set in record_sets:
    print(f"Loading records for RecordSet {record_set} ...")
    records = list(dataset.records(record_set=record_set))
    df = pd.DataFrame(records)
    dataframes[record_set] = df
    print(f"  Columns for RecordSet {record_set}: {df.columns.tolist()}")
    print(f"  First 3 records:\n{df.head(3)}\n")
# For demonstration, select the first record set
if len(record_sets):
    selected_record_set = record_sets[0]
    print(f"Sample columns: {dataframes[selected_record_set].columns.tolist()}")
    display(dataframes[selected_record_set].head(5))

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Set record set and field IDs after reviewing the previous overview
record_set_id = selected_record_set

df = dataframes[record_set_id]
print(f"Available columns: {df.columns.tolist()}")

# Try to select a numeric field for analysis
# User can adjust this field based on available columns. For demonstration, attempt to infer a likely numeric column.
numeric_candidates = [
    col for col in df.columns if col.lower() in ['mw', 'molecular_weight', 'peptide_count', 'abundance', 'coverage'] or df[col].dtype in ['int64', 'float64']
]
if numeric_candidates:
    numeric_field = numeric_candidates[0]
else:
    numeric_field = df.select_dtypes(include=['float64', 'int64']).columns[0] if not df.select_dtypes(include=['float64', 'int64']).empty else df.columns[0]

print(f"Selected numeric field for filtering: {numeric_field}")
threshold = df[numeric_field].mean() if pd.api.types.is_numeric_dtype(df[numeric_field]) else 10

# Filter out records where numeric_field > threshold
if pd.api.types.is_numeric_dtype(df[numeric_field]):
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
    display(filtered_df.head())

    # Normalize
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
    # Find a group field
    group_candidates = [col for col in df.columns if col.lower() in ['description','accession','group','protein_group']]
    if group_candidates:
        group_field = group_candidates[0]
        grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
        print(f"Grouped data by {group_field} (showing mean values):")
        display(grouped_df.head())
else:
    print(f"Field '{numeric_field}' could not be treated as numeric. Skipping EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if pd.api.types.is_numeric_dtype(df[numeric_field]):
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.show()

    if 'description' in df.columns:
        plt.figure(figsize=(10,5))
        sns.boxplot(x='description', y=numeric_field, data=df)
        plt.xticks(rotation=90)
        plt.title(f"{numeric_field} by description")
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

<!-- Add your findings here after analysis: For example, main trends, field distributions, or any insights from the data -->